In [1]:
%%writefile lexer.c

#include <stdio.h>
#include <string.h>
#include <ctype.h>

FILE *fp;

char delim[14] = {
    ' ', '\t', '\n', ',', ';', '(', ')',
    '{', '}', '[', ']', '#', '<', '>'
};

char oper[7] = {
    '+', '-', '*', '/', '%', '=', '!'
};

char key[21][12] = {
    "int", "float", "char", "double", "bool",
    "void", "extern", "unsigned", "goto", "static",
    "class", "struct", "for", "if", "else",
    "return", "register", "long", "while", "do"
};

char predirect[2][12] = {
    "include", "define"
};

char header[6][15] = {
    "stdio.h", "conio.h", "malloc.h",
    "process.h", "string.h", "ctype.h"
};

int isdelim(char);
int isop(char);
void check(char[]);
void analyze();
void skipcomment();

int fop = 0, numflag = 0, f = 0;
char c, ch, sop;

int main()
{
    char fname[50];

    printf("\nEnter filename: ");
    scanf("%49s", fname);

    fp = fopen(fname, "r");

    if (fp == NULL)
    {
        printf("\nThe file doesn't exist.\n");
        return 1;
    }

    analyze();

    fclose(fp);

    printf("\n\nEnd of file\n");

    return 0;
}

void analyze()
{
    char token[50];
    int j = 0;

    while ((c = getc(fp)) != EOF)
    {
        if (c == '/')
        {
            ch = getc(fp);

            if (ch == '/' || ch == '*')
            {
                ungetc(ch, fp);
                skipcomment();
            }
            else
            {
                ungetc(ch, fp);

                if (j > 0)
                {
                    token[j] = '\0';
                    check(token);
                    j = 0;
                }

                printf("\nOperator\t /");
            }
        }

        else if (c == '"')
        {
            while ((c = getc(fp)) != '"' && c != EOF);
        }

        else if (isalpha((unsigned char)c) || c == '_')
        {
            token[j++] = c;

            while ((c = getc(fp)) != EOF &&
                   (isalnum((unsigned char)c) || c == '_'))
            {
                token[j++] = c;
            }

            token[j] = '\0';
            check(token);
            j = 0;

            if (c != EOF)
                ungetc(c, fp);
        }

        else if (isdigit((unsigned char)c))
        {
            token[j++] = c;

            while ((c = getc(fp)) != EOF &&
                   (isdigit((unsigned char)c) || c == '.'))
            {
                token[j++] = c;
            }

            token[j] = '\0';

            printf("\nNumber\t\t %s", token);

            j = 0;

            if (c != EOF)
                ungetc(c, fp);
        }

        else if (isdelim(c))
        {
            printf("\nDelimiter\t %c", c);
        }

        else if (isop(c))
        {
            printf("\nOperator\t %c", c);
        }
    }
}

int isdelim(char c)
{
    int i;

    for (i = 0; i < 14; i++)
    {
        if (c == delim[i])
            return 1;
    }

    return 0;
}

int isop(char c)
{
    int i;

    for (i = 0; i < 7; i++)
    {
        if (c == oper[i])
            return 1;
    }

    return 0;
}

void check(char t[])
{
    int i;

    for (i = 0; i < 2; i++)
    {
        if (strcmp(t, predirect[i]) == 0)
        {
            printf("\nPreprocessor directive\t%s", t);
            return;
        }
    }

    for (i = 0; i < 6; i++)
    {
        if (strcmp(t, header[i]) == 0)
        {
            printf("\nHeader file\t\t%s", t);
            return;
        }
    }

    for (i = 0; i < 21; i++)
    {
        if (strcmp(key[i], t) == 0)
        {
            printf("\nKeyword\t\t\t%s", t);
            return;
        }
    }

    printf("\nIdentifier\t\t%s", t);
}

void skipcomment()
{
    ch = getc(fp);

    if (ch == '/')
    {
        while ((ch = getc(fp)) != '\n' && ch != EOF);
    }

    else if (ch == '*')
    {
        while ((ch = getc(fp)) != EOF)
        {
            if (ch == '*')
            {
                ch = getc(fp);

                if (ch == '/')
                    break;
            }
        }
    }
}

Writing lexer.c


In [2]:
%%writefile input.c

#include <stdio.h>

int main()
{
    int a = 10;
    float b = 20.5;

    // This is a comment

    if (a > 5)
    {
        printf("Hello World");
    }

    return 0;
}

Writing input.c


In [3]:
!gcc lexer.c -o lexer

In [4]:
!printf "input.c\n" | ./lexer


Enter filename: 
Delimiter	 

Delimiter	 #
Preprocessor directive	include
Delimiter	  
Delimiter	 <
Identifier		stdio
Identifier		h
Delimiter	 >
Delimiter	 

Delimiter	 

Keyword			int
Delimiter	  
Identifier		main
Delimiter	 (
Delimiter	 )
Delimiter	 

Delimiter	 {
Delimiter	 

Delimiter	  
Delimiter	  
Delimiter	  
Delimiter	  
Keyword			int
Delimiter	  
Identifier		a
Delimiter	  
Operator	 =
Delimiter	  
Number		 10
Delimiter	 ;
Delimiter	 

Delimiter	  
Delimiter	  
Delimiter	  
Delimiter	  
Keyword			float
Delimiter	  
Identifier		b
Delimiter	  
Operator	 =
Delimiter	  
Number		 20.5
Delimiter	 ;
Delimiter	 

Delimiter	 

Delimiter	  
Delimiter	  
Delimiter	  
Delimiter	  
Delimiter	 

Delimiter	  
Delimiter	  
Delimiter	  
Delimiter	  
Keyword			if
Delimiter	  
Delimiter	 (
Identifier		a
Delimiter	  
Delimiter	 >
Delimiter	  
Number		 5
Delimiter	 )
Delimiter	 

Delimiter	  
Delimiter	  
Delimiter	  
Delimiter	  
Delimiter	 {
Delimiter	 

Delimiter	  
Delimiter	  
Delimiter	  
D